# ICDN Architecture Stress Test

Controlled computational stress test of the ICDN architecture across
(n, k) combinations. Modules are instantiated with the best-trial
hyperparameters (`best_trial_params.json`) and random weights, since
forward-pass latency/memory depends on tensor shapes, not weight values.
All results are written to a single CSV (`results/stress_test_icdn.csv`);
the paper panels are just filtered views of that one table.

In [14]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from src.nn.context.context_mlp import SharedProductEncoder
from src.nn.heads.parameter_head import DemandParameterHead
from src.nn.heads.neighbor_selector import SparseNeighborSelector
from src.nn.spline import MultiCubicSplineBasis
from src.multiproduct.context import ProductTokenBuilder

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(Path("../results/best_trial_params.json")) as f:
    best_trial = json.load(f)["params"]

HIDDEN_OPTIONS = {
    "64_32": (64, 32), "128_64": (128, 64), "192_96": (192, 96),
    "256_128": (256, 128), "256_128_64": (256, 128, 64),
}

# From Optuna (best_trial_params.json):
K_SPLINES  = int(best_trial["N_KNOTS"])
HIDDEN     = HIDDEN_OPTIONS[best_trial["HIDDEN_KEY"]]
BATCH_SIZE = int(best_trial["BATCH_SIZE"])

# Constants NOT searched by Optuna, fixed manually in nn_final_evaluation.ipynb:
ACT        = "gelu"
D_ATTN     = 16
D_STORE, D_BRAND, D_STYLE = 16, 8, 8
N_BASELINE = 5   # real n evaluated in the paper
K_BASELINE = 4   # real K_NEIGHBORS evaluated in the paper

H = HIDDEN[-1]   # encoder output dimension

print(f"K_SPLINES={K_SPLINES}  HIDDEN={HIDDEN}  H={H}  D_ATTN={D_ATTN}")

K_SPLINES=3  HIDDEN=(256, 128, 64)  H=64  D_ATTN=16


# Architecture + shared synthetic-context builder

In [15]:
# Forward-pass latency/memory cost depends on tensor SHAPES, not weight values,
# so it's enough to instantiate the shared modules once with random weights.
torch.manual_seed(0)

param_head = DemandParameterHead(
    hidden_dim=H, K_splines=K_SPLINES, n=N_BASELINE, use_cross=True
).eval().to(device)

_dummy_tb = ProductTokenBuilder(n=1, n_stores=1, d_store=D_STORE,
                                 n_brands=1, d_brand=D_BRAND, n_styles=1, d_style=D_STYLE)
D_TOKEN = _dummy_tb.d_token
encoder = SharedProductEncoder(d_in=D_TOKEN, hidden=HIDDEN, act=ACT, dropout=0.0).eval().to(device)

def make_selector(k_neighbors: int) -> SparseNeighborSelector:
    return SparseNeighborSelector(d_hidden=H, d_attn=D_ATTN, k_neighbors=k_neighbors).eval().to(device)

def synth_h(n: int, batch_size: int) -> torch.Tensor:
    return torch.randn(batch_size, n, H, device=device)

def synth_meta(n: int, n_categories: int = 3):
    """Synthetic category/brand/style/liters, grouped so the top-k selector has
    real 'same category' structure to exploit."""
    category = torch.arange(n, device=device) % n_categories
    brand    = torch.arange(n, device=device) % max(2, n_categories)
    style    = torch.arange(n, device=device) % max(2, n_categories)
    liters   = torch.empty(n, device=device).uniform_(0.3, 2.0)
    return category, brand, style, liters

def synth_splines(n: int, K: int) -> MultiCubicSplineBasis:
    q = torch.linspace(0.05, 0.95, K)
    base_knots = torch.distributions.Normal(0, 1).icdf(q)
    knots = base_knots.unsqueeze(0).repeat(n, 1)
    return MultiCubicSplineBasis(knots=knots, shift=torch.zeros(n), scale=torch.ones(n)).to(device)

def build_synthetic_context(n: int, k: int):
    """All (n, k)-dependent synthetic assets shared by every benchmark function."""
    k_eff = min(k, n - 1)
    category, brand, style, liters = synth_meta(n)
    splines = synth_splines(n, K_SPLINES)
    selector = make_selector(k_eff)
    return k_eff, category, brand, style, liters, splines, selector

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

n_params = {
    "encoder":           count_params(encoder),
    "param_head":        count_params(param_head),
    "neighbor_selector": count_params(make_selector(K_BASELINE)),
}
print("Shared, n/k-agnostic parameter counts:", n_params)

Shared, n/k-agnostic parameter counts: {'encoder': 58560, 'param_head': 2002, 'neighbor_selector': 2051}


# GPU timer

In [16]:
class GpuTimer:
    def __init__(self, device, warmup=3, repeats=15):
        self.device, self.warmup, self.repeats = device, warmup, repeats

    def time_block(self, fn):
        """Runs fn() warmup+repeats times; returns (median_ms, peak_mb, last_result)
        so callers never need to call fn() again just to read its output."""
        result = None
        for _ in range(self.warmup):
            result = fn()
        if self.device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        times = []
        for _ in range(self.repeats):
            if self.device == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            result = fn()
            if self.device == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000.0)
        peak_mb = (torch.cuda.max_memory_allocated() / 1e6) if self.device == "cuda" else float("nan")
        return float(np.median(times)), peak_mb, result

timer = GpuTimer(device)

# Bench one

In [17]:
@torch.no_grad()
def bench_one(n: int, k: int, batch_size: int = BATCH_SIZE):
    k_eff, category, brand, style, liters, splines, selector = build_synthetic_context(n, k)

    # ── Candidate scoring: dense (n,n) score matrix (q·k + meta_bonus) ──
    h_score_sample = synth_h(n, batch_size=256)
    scoring_ms, mem1, mean_scores = timer.time_block(
        lambda: selector.accumulate_mean_scores([h_score_sample], category, brand, style, liters)
    )

    # ── Top-k: edge selection from the score matrix ──
    not_self, same_cat, meta_bonus = selector._meta_bonus(category, brand, style, liters, device)
    topk_ms, mem2, pairs = timer.time_block(
        lambda: selector._build_pairs(mean_scores, not_self, same_cat)
    )
    i_idx, j_idx = pairs[0], pairs[1]

    # Freeze the graph: from here on, cost is O(n*k_eff), not O(n^2).
    selector.frozen_pairs = pairs
    selector.frozen_edge_bonus = meta_bonus[i_idx, j_idx]

    h = synth_h(n, batch_size)
    x = torch.randn(batch_size, n, device=device) * 0.1
    Bx, dBx, _ = splines(x)

    # ── Parameter/attention + own-price terms, O(n) ──
    def own_terms():
        _, attn = selector.run(h, category, brand, style, liters)
        params = param_head.run(h, pairs=pairs)
        b, beta, w = params["b"], params["beta"], params["w"]
        y_hat   = b + beta * x + (w * Bx).sum(dim=-1)
        eps_hat = beta + (w * dBx).sum(dim=-1)
        return params, attn, eps_hat
    demand_ms, mem3, (params, attn, eps_hat) = timer.time_block(own_terms)

    # ── Selected cross-interaction terms, O(n*k_eff) ──
    Bx_i, Bx_j   = Bx[:, i_idx, :],  Bx[:, j_idx, :]
    dBx_i, dBx_j = dBx[:, i_idx, :], dBx[:, j_idx, :]
    x_j = x[:, j_idx]

    def interaction_eval():
        contrib_yi = (
            params["beta_cross"] * x_j
            + (params["w_cross"] * Bx_j).sum(dim=-1)
            + torch.einsum('bpk,bpkl,bpl->bp', Bx_i, params["u"], Bx_j)
        ) * attn
        contrib_ei = torch.einsum('bpk,bpkl,bpl->bp', dBx_i, params["u"], Bx_j) * attn
        return contrib_yi, contrib_ei
    interaction_ms, mem4, _ = timer.time_block(interaction_eval)

    # ── Dense elasticity materialization: full E (B,n,n) — incremental O(n^2) memory ──
    def elasticity_full():
        B = h.shape[0]
        E = torch.zeros(B, n, n, device=device)
        E[:, torch.arange(n), torch.arange(n)] = eps_hat
        E_cross = (
            params["beta_cross"]
            + (params["w_cross"] * dBx_j).sum(dim=-1)
            + torch.einsum('bpk,bpkl,bpl->bp', Bx_i, params["u"], dBx_j)
        ) * attn
        E[:, i_idx, j_idx] = E_cross
        return E
    elasticity_ms, mem5, _ = timer.time_block(elasticity_full)

    return {
        "n": n, "k": k, "k_eff": k_eff,
        "Graph density": round(k_eff / (n - 1), 4) if n > 1 else float("nan"),
        "Candidate scoring per batch (ms)": round(scoring_ms, 4),
        "Top-k ms": round(topk_ms, 4),
        "Parameter/attention + own-term ms": round(demand_ms, 4),
        "Selected cross-interaction ms": round(interaction_ms, 4),
        "Dense elasticity materialization ms": round(elasticity_ms, 4),
        "Peak GPU MB": round(max(mem1, mem2, mem3, mem4, mem5), 2),
    }

# Bench Training Step

In [18]:
def bench_training_step(n: int, k: int, batch_size: int = BATCH_SIZE):
    # frozen_pairs stays None on purpose: real training never freezes the graph —
    # it always recomputes candidate scoring + top-k every batch; freeze_graph()
    # is called only ONCE, after training, before eval.
    _, category, brand, style, liters, splines, selector = build_synthetic_context(n, k)

    trainable = list(param_head.parameters()) + list(selector.parameters())
    optimizer = torch.optim.Adam(trainable, lr=1e-3)

    x = torch.randn(batch_size, n, device=device) * 0.1
    Bx, dBx, _ = splines(x)
    y_target = torch.randn(batch_size, n, device=device)

    def training_step():
        optimizer.zero_grad()
        h = synth_h(n, batch_size)
        pairs, attn = selector.run(h, category, brand, style, liters)
        i_idx, j_idx = pairs[0], pairs[1]
        params = param_head.run(h, pairs=pairs)

        b, beta, w = params["b"], params["beta"], params["w"]
        y_hat = b + beta * x + (w * Bx).sum(dim=-1)

        Bx_i, Bx_j = Bx[:, i_idx, :], Bx[:, j_idx, :]
        contrib_yi = (
            params["beta_cross"] * x[:, j_idx]
            + (params["w_cross"] * Bx_j).sum(dim=-1)
            + torch.einsum('bpk,bpkl,bpl->bp', Bx_i, params["u"], Bx_j)
        ) * attn
        y_hat = y_hat.scatter_add(1, i_idx.unsqueeze(0).expand(batch_size, -1), contrib_yi)

        loss = torch.nn.functional.mse_loss(y_hat, y_target)
        loss.backward()
        optimizer.step()

    step_ms, peak_mb, _ = timer.time_block(training_step)
    return step_ms, peak_mb

# Stress Test

In [ ]:
n_values = [5, 10, 25, 50, 100, 200]
k_values = [1, 2, 4, 8, 16, 32]
grid = sorted({(n, k) for n in n_values for k in k_values if min(k, n - 1) == k} |
              {(n, n - 1) for n in n_values})  # always include the dense case k=n-1

rows = []
for n, k in grid:
    row = bench_one(n, k)
    step_ms, peak_train_mb = bench_training_step(n, k)
    row["Training step ms"] = round(step_ms, 4)
    row["Peak training GPU MB"] = round(peak_train_mb, 2)
    rows.append(row)

df_stress = pd.DataFrame(rows).sort_values(["n", "k"]).reset_index(drop=True)

# ── Costs of different frequency: one-off graph construction vs. recurring evaluation ──
df_stress["Graph construction ms"] = (
    df_stress["Candidate scoring per batch (ms)"] + df_stress["Top-k ms"]
)
df_stress["Frozen-graph evaluation ms (sparse)"] = (
    df_stress["Parameter/attention + own-term ms"] + df_stress["Selected cross-interaction ms"]
)
df_stress["Frozen-graph evaluation ms (+ dense E)"] = (
    df_stress["Frozen-graph evaluation ms (sparse)"] + df_stress["Dense elasticity materialization ms"]
)

# ── Shared parameter counts: constant across every (n, k) row — see cell 2 ──
df_stress["Shared params (encoder)"]    = n_params["encoder"]
df_stress["Shared params (param_head)"] = n_params["param_head"]
df_stress["Shared params (selector)"]   = n_params["neighbor_selector"]

OUT_PATH = Path("../results/stress_test_icdn.csv")
df_stress.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}  ({len(df_stress)} rows, {df_stress.shape[1]} columns)")

Saved to ../results/stress_test_icdn.csv  (35 rows, 18 columns)


,n,k,k_eff,Graph density,Candidate scoring per batch (ms),Top-k ms,Parameter/attention + own-term ms,Selected cross-interaction ms,Dense elasticity materialization ms,Peak GPU MB,Training step ms,Peak training GPU MB,Graph construction ms,Frozen-graph evaluation ms (sparse),Frozen-graph evaluation ms (+ dense E),Shared params (encoder),Shared params (param_head),Shared params (selector)
0,5,1,1,0.2500,0.2945,0.0758,0.2864,0.0898,0.1276,20.58,1.6986,20.36,0.3703,0.3762,0.5038,58560,2002,2051
1,5,2,2,0.5000,0.2587,0.0821,0.2766,0.0893,0.1198,22.70,1.6725,22.40,0.3408,0.3659,0.4857,58560,2002,2051
2,5,4,4,1.0000,0.2564,0.0818,0.2747,0.0893,0.1191,25.61,1.6648,25.16,0.3382,0.3640,0.4831,58560,2002,2051
3,10,1,1,0.1111,0.2522,0.0689,0.2636,0.0891,0.1183,23.46,1.6454,22.97,0.3211,0.3527,0.4710,58560,2002,2051
4,10,2,2,0.2222,0.2476,0.0845,0.2664,0.0913,0.1175,26.37,1.6573,25.74,0.3321,0.3577,0.4752,58560,2002,2051
5,10,4,4,0.4444,0.2490,0.0828,0.2743,0.0962,0.1185,31.53,1.6637,30.61,0.3318,0.3705,0.4890,58560,2002,2051
6,10,8,8,0.8889,0.2515,0.0815,0.2604,0.1291,0.1330,43.16,1.6731,41.67,0.3330,0.3895,0.5225,58560,2002,2051
7,10,9,9,1.0000,0.2517,0.0831,0.2616,0.1373,0.1386,47.51,1.6613,45.88,0.3348,0.3989,0.5375,58560,2002,2051
8,25,1,1,0.0417,0.2504,0.0699,0.2627,0.0902,0.1202,29.79,1.6529,28.52,0.3203,0.3529,0.4731,58560,2002,2051
9,25,2,2,0.0833,0.2534,0.0831,0.2635,0.1026,0.1196,37.06,1.6700,35.43,0.3365,0.3661,0.4857,58560,2002,2051
